In [2]:
####################################
#ENVIRONMENT SETUP

In [3]:
#LIBRARIES

#system
import os
import sys

#data classes
import xarray as xr

#loading bar
from tqdm import tqdm

In [4]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [5]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data", "TRACER")
dataType = "RadarData"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/Observation_Data/TRACER/RadarData



In [6]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [7]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

# RunType = ("TRACER","MOIST","NSSL")
RunType = ("TRACER","DRY","NSSL")
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 289/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL/history_cartesian/history.2022-06-08_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL/diag_cartesian/diag.2022-06-08_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           DRY
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-08 to 2022-06-11
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:289
 # Diag Files:   289
 # Time Steps:   289
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/DRY/MPAS-Model_NSSL
 Static File:    TRACER_regional5250_scaled3_x20.835586.static.latlon.nc



In [8]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [10]:
########################
#CODE INFORMATION

In [11]:
#Getting Data
# https://projectpythia.org/mrms-cookbook/notebooks/ch4-realtimedata/
# Authors:
# Ty Janoski
# City College of New York and NOAA/OAR National Severe Storms Laboratory
# Mya Sears
# NSF National Center for Atmospheric Research
# Bella Condo
# University at Albany (State University of New York)
# JD Heaton
# Metropolitan State University of Denver
# MaKenna Collins
# Jackson State University
# Maxwell Grover
# Argonne National Laboratory

In [12]:
########################
#LIBRARIES

In [13]:
#system packages
import re

#data loading packages
import pandas as pd

# Packages required to request and open data from AWS S3
import s3fs
import urllib
import tempfile
import gzip

In [14]:
########################
#DATA RETRIEVAL FUNCTIONS

In [18]:
# ============================================================
# Helper: extract time from filename
# ============================================================
def extract_time_from_filename(fname):
    """Parse MRMS filename and return a pandas.Timestamp (UTC)."""
    match = re.search(r"_(\d{8}-\d{6})\.grib2\.gz", fname)
    return pd.to_datetime(match.group(1), format="%Y%m%d-%H%M%S", utc=True)

# ============================================================
# Select target times at 6-hour intervals
# ============================================================
def select_nearest_to_targets(files, interval_hours="6h"):
    """Return subset of file paths nearest to each N-hour UTC time."""
    # Extract actual timestamps from filenames
    file_times = pd.Series({f: extract_time_from_filename(f) for f in files}).sort_values()

    # Build list of desired UTC times covering file range
    start = file_times.min().floor(interval_hours)   # lowercase h
    end   = file_times.max().ceil(interval_hours)
    targets = pd.date_range(start, end, freq=interval_hours, tz="UTC")

    selected = []
    for t in targets:
        nearest_idx = (abs(file_times - t)).argmin()
        selected.append(file_times.index[nearest_idx])

    return selected

# def RetrieveMRMSRadarData_V1(ModelData,region,product,datestrings, 
#                           interval_hours="6h",
#                           outputPath=""):
#     """
#     Retrieve MRMS reflectivity data and save each selected timestep
#     as an combined NetCDF file.
#     """
#     aws = s3fs.S3FileSystem(anon=True)
    
#     all_datasets = []
#     for datestring in datestrings:
#         print(f"\n=== Processing {datestring} ===")
    
#         # List all files for this date
#         try:
#             data_files = aws.ls(f'noaa-mrms-pds/{region}/{product}/{datestring}/')
#         except Exception as e:
#             print(f"Could not access {datestring}: {e}")
#             continue
    
#         if not data_files:
#             print(f"No files found for {datestring}")
#             continue
    
#         data_files = sorted(data_files)
    
#         # (Optional) Filter to specific times or interval
#         selected_files = select_nearest_to_targets(data_files, interval_hours=interval_hours)
    
#         datasets = []
#         for f in selected_files:
#             print(f'working on {f}')
#             try:
#                 response = urllib.request.urlopen(f"https://noaa-mrms-pds.s3.amazonaws.com/{f[14:]}")
#                 compressed_file = response.read()
    
#                 with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
#                     tmp.write(gzip.decompress(compressed_file))
#                     tmp.flush()
#                     ds = xr.load_dataarray(tmp.name, engine="cfgrib", decode_timedelta=True)
    
#                     # Drop differing coords
#                     for coord in ["valid_time", "step"]:
#                         if coord in ds.coords:
#                             ds = ds.drop_vars(coord)
    
#                     # Attach true timestamp from filename
#                     file_time = extract_time_from_filename(f)
#                     ds = ds.expand_dims(time=[file_time])
    
#                     # Subset
#                     subset = DataSubsetting_Class.SubsetDataRegion(ds, ModelData)
#                     datasets.append(subset)
    
#             except Exception as e:
#                 print(f"Failed to load {f}: {e}")
#                 continue
    
#         if datasets:
#             day_data = xr.concat(datasets, dim="time", coords="minimal")
#             all_datasets.append(day_data)
    
#     # ============================================================
#     # Combine all dates into one dataset
#     # ============================================================
#     if all_datasets:
#         data_multi = xr.concat(all_datasets, dim="time", coords="minimal")
#         print("\nCombined dataset shape:", data_multi.shape)
#     else:
#         print("No datasets loaded.")


#     # ============================================================
#     # Saving Data to Single NetCDF Dataset
#     # ============================================================
#     times = pd.to_datetime(data_multi.time.to_pandas()).tz_localize(None)
#     data_multi = data_multi.assign_coords(time=times)
    
#     # Now you can save safely
#     outputFile = f"MRMSReflectivity_{region}_{ModelData.region}_{datestrings[0]}-{datestrings[-1]}.nc"
#     outputFilePath = os.path.join(outputPath, outputFile)
#     data_multi.to_netcdf(outputFilePath)
#     return data_multi

def RetrieveMRMSRadarData_V2(ModelData, region, product, datestrings,
                             interval_hours="6h",
                             outputPath=""):
    """
    Retrieve MRMS reflectivity data and save each selected timestep
    as an individual NetCDF file.
    """
    aws = s3fs.S3FileSystem(anon=True)

    for datestring in datestrings:
        print(f"\n=== Processing {datestring} ===")

        # List all files for this date
        try:
            data_files = aws.ls(f'noaa-mrms-pds/{region}/{product}/{datestring}/')
        except Exception as e:
            print(f"Could not access {datestring}: {e}")
            continue

        if not data_files:
            print(f"No files found for {datestring}")
            continue

        data_files = sorted(data_files)
        selected_files = select_nearest_to_targets(data_files, interval_hours=interval_hours)

        for f in tqdm(selected_files, desc=f"{datestring} files", leave=False, unit="file"):

            try:
                response = urllib.request.urlopen(f"https://noaa-mrms-pds.s3.amazonaws.com/{f[14:]}")
                compressed_file = response.read()

                with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
                    tmp.write(gzip.decompress(compressed_file))
                    tmp.flush()
                    ds = xr.load_dataarray(tmp.name, engine="cfgrib", decode_timedelta=True)
                    ds.name = product

                    # Drop differing coords
                    for coord in ["valid_time", "step"]:
                        if coord in ds.coords:
                            ds = ds.drop_vars(coord)

                    # Attach true timestamp from filename
                    file_time = extract_time_from_filename(f)
                    ds = ds.expand_dims(time=[file_time])

                    # Subset
                    subset = DataSubsetting_Class.SubsetDataRegion(ds, ModelData)

                    # Convert to timezone-naive time
                    subset = subset.assign_coords(
                        time=pd.to_datetime(subset.time.to_pandas()).tz_localize(None)
                    )

                    # --- Save each timestep as its own NetCDF ---
                    time_str = file_time.strftime("%Y%m%d-%H%M%S")
                    outputFile = f"MRMSReflectivity_{region}_{ModelData.region}_{time_str}.nc"
                    os.makedirs(outputPath, exist_ok=True)
                    outputFilePath = os.path.join(outputPath, outputFile)

                    subset.to_netcdf(outputFilePath)
                    print(f"Saved: {outputFilePath}")

            except Exception as e:
                print(f"Failed to load {f}: {e}")
                continue

In [19]:
########################
#DATA RETRIEVAL

In [ ]:
# ============================================================
# Main Loop: Load multiple dates
# ============================================================
region_options = [
    "CONUS",
    "ALASKA",
    "CARIB",
    "GUAM",
    "HAWAII"
]

product_options = ["MergedReflectivityQC_01.00"]

# Retrieve the user selection from 'Region' 
region = "CONUS" if ModelData.region=="TRACER" else "Hawaii"

# Retrieve the user selection from 'MRMS product'
product = product_options[0]
datestrings = [datestring.replace('-','') for datestring in ModelData.simulationDates[:-1]]

MRMSData = RetrieveMRMSRadarData_V2(ModelData,region,product,datestrings,
                                 interval_hours="15min",
                                 outputPath = os.path.join(DirectoryManager.dataDirectory,
                                                           "Observation_Data/TRACER/MRMS_RadarData",
                                                           f"{ModelData.simulationDates[0]}_{ModelData.simulationDates[-1]}"))